# PCA Repropucibility in Python

## Packages

Before first time, run this on the terminal in your python enviroment:

```bash
source activate general
pip install -r requirements.txt
```

Here `general` is the name of the enviroment created as per ScienceCluster instruction:

```bash
$ module load miniforge3
$ mamba create -n general ipykernel
$ source activate general
$ ipython kernel install --user --name general
```

In [ ]:
# general
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# scanpy
# Core scverse libraries
from __future__ import annotations
import anndata as ad
# Data retrieval
import pooch
import scanpy as sc

## PCA Tutorial

Use the PCA implementation in [sciit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html).

In [ ]:
X = np.array([[-1, -1], [-2, -1], [-3, -2], [1, 1], [2, 1], [3, 2]])
pca = PCA(n_components=2, svd_solver='randomized')
pca.fit(X)
print(pca.explained_variance_ratio_)
print(pca.singular_values_)

## Clustering Tutorial

Clustering [tutorial](https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html) from `scanpy`.

Load example data.

In [ ]:
EXAMPLE_DATA = pooch.create(
    path=pooch.os_cache("scverse_tutorials"),
    base_url="doi:10.6084/m9.figshare.22716739.v1/",
)
EXAMPLE_DATA.load_registry_from_doi()

In [ ]:
samples = {
    "s1d1": "s1d1_filtered_feature_bc_matrix.h5",
    "s1d3": "s1d3_filtered_feature_bc_matrix.h5",
}
adatas = {}

for sample_id, filename in samples.items():
    path = EXAMPLE_DATA.fetch(filename)
    sample_adata = sc.read_10x_h5(path)
    sample_adata.var_names_make_unique()
    adatas[sample_id] = sample_adata

adata = ad.concat(adatas, label="sample")
adata.obs_names_make_unique()
print(adata.obs["sample"].value_counts())
adata

In [ ]:
sc.settings.set_figure_params(dpi=50, facecolor="white")

Quality control.

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
sc.pp.scrublet(adata, batch_key="sample")

Normalization.

In [ ]:
# Saving count data
adata.layers["counts"] = adata.X.copy()

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)

Feature selection.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key="sample")

In [ ]:
sc.pl.highly_variable_genes(adata)

Dimensionality reduction.

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
sc.pl.pca(
    adata,
    color=["sample", "sample", "pct_counts_mt", "pct_counts_mt"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=2,
)

Nearest neighbor graph construction and visualization.

In [ ]:
sc.pp.neighbors(adata)

In [ ]:
sc.tl.umap(adata)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))
sc.pl.umap(
    adata,
    color="sample",
    # Setting a smaller point size to get prevent overlap
    size=30,
    ax=ax,
    show=True
)

Clustering.

In [ ]:
# Using the igraph implementation and a fixed number of iterations can be significantly faster,
# especially for larger datasets
sc.tl.leiden(adata, flavor="igraph", n_iterations=2)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))
sc.pl.umap(
    adata, 
    color=["leiden"],
    size=30,
    ax=ax,
    show=True
)

## Session Info

In [ ]:
%load_ext watermark
%watermark
!lscpu | grep 'Model name'